# 🐚 Pearlhash Miner - Google Colab

**⚠️ T4 GPU warning:** Pearlhash needs compute ≥ 8.0. T4 is 7.5.
If you get T4, mining won't produce hashes.

**Fix:** Runtime → Change runtime type → **L4** or **A100**

> If L4/A100 is unavailable, just keep trying — Colab Free sometimes assigns them.

In [ ]:
#@title 1️⃣ Check GPU
import subprocess

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f'GPU: {gpu}')

if gpu:
    cc = gpu.split(',')[1].strip()
    major = int(cc.split('.')[0])
    if major < 8:
        print('\n❌ Compute {cc} < 8.0 — Pearlhash needs compute 8.0+')
        print('   Go to: Runtime → Change runtime type → L4 or A100')
        print('   (Or just run anyway — Colab might have assigned a better GPU next time)')
    else:
        print(f'\n✅ Compute {cc} — good to go!')
else:
    print('\n❌ No GPU detected. Enable GPU: Runtime → Change runtime type → GPU')

In [ ]:
#@title 2️⃣ Download Pearlhash Miner
import os

!curl -sL https://pearlhash.xyz/downloads/pearl-miner-v11 -o /content/pearl-miner
!chmod +x /content/pearl-miner

size = os.path.getsize('/content/pearl-miner')
print(f'✅ Pearlhash miner downloaded ({size/1024/1024:.1f} MB)')
print(f'   Version: v11')

In [ ]:
#@title 3️⃣ START MINING 🚀
import subprocess, signal, time

WALLET = 'prl1p5jh9h20cgku4r07ucqvyjvkh36956k45vh5dkh80luwwfrpw7ykslnga85'
WORKER = 'colab-gpu'
POOL = '84.32.220.219:9000'

print(f'🐚 Pearlhash Miner')
print(f'   Wallet: {WALLET[:15]}...{WALLET[-8:]}')
print(f'   Worker: {WORKER}')
print(f'   Pool:   {POOL}')
print('='*50)

proc = subprocess.Popen(
    ['/content/pearl-miner', '--host', POOL, '--user', WALLET, '--worker', WORKER],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

try:
    start = time.time()
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line:
            print(line.rstrip(), flush=True)
    # Read remaining output
    out = proc.stdout.read()
    if out:
        print(out)
    if proc.returncode != 0:
        print(f'\n❌ Miner exited with code {proc.returncode}')
        if proc.returncode == -11:
            print('SEGFAULT — GPU kernel not compatible.')
            print('Change runtime: Runtime → Change runtime type → L4 or A100')
except KeyboardInterrupt:
    proc.send_signal(signal.SIGINT)
    proc.wait(timeout=30)
    print(f'\n✅ Stopped after {int(time.time()-start)}s')